In [55]:
%cd drive/MyDrive/Colab\ Notebooks/ML_Prac/lesson6-transformer

[Errno 2] No such file or directory: 'drive/MyDrive/Colab Notebooks/ML_Prac/lesson6-transformer'
/content/drive/MyDrive/Colab Notebooks/ML_Prac/lesson6-transformer


layers

---



In [56]:
import torch
import math
from copy import deepcopy
from typing import Union, Optional, List, Tuple, Callable, Any
import os


class PositionalEncoding(torch.nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000) -> None:
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(2 * torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1), :]
        return x

class Embedding(torch.nn.Module):
    def __init__(self, d_model: int, vocab_len: int, pad_index: int) -> None:
        super().__init__()
        self.d_model = d_model
        self.embedding = torch.nn.Embedding(vocab_len, self.d_model, padding_idx=pad_index)
        self.positional_encoding = PositionalEncoding(self.d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.embedding(x)
        x = self.positional_encoding(x)
        return x

class ScaledDotProductAttention(torch.nn.Module):
    def __init__(self, d_model: int) -> None:
        super().__init__()
        self.d_model = d_model
        self.factor = math.sqrt(self.d_model)

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        scores = torch.matmul(query, key.transpose(-2, -1)) / self.factor
        if mask is not None:
            scores = scores.masked_fill(mask == 0, torch.finfo(scores.dtype).min)
        attention_weights = torch.nn.Softmax(dim=-1)(scores)
        out = torch.matmul(attention_weights, value)
        return out, attention_weights

class MultiheadAttention(torch.nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        self.q_linear = torch.nn.Linear(d_model, d_model)
        self.k_linear = torch.nn.Linear(d_model, d_model)
        self.v_linear = torch.nn.Linear(d_model, d_model)
        self.out_linear = torch.nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention(d_model)

        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        batch_size = query.size(0)

        query = self.q_linear(query)
        query = query.view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        key = self.k_linear(key)
        key = key.view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        value = self.v_linear(value)
        value = value.view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        if mask is not None:
            mask = mask.unsqueeze(1)
        x, attention_weights = self.attention(query, key, value, mask)

        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        x = self.out_linear(x)
        x = self.dropout(x)
        return x, attention_weights

class FeedForward(torch.nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.d_model = d_model
        self.d_ff = d_ff
        if d_ff % d_model:
            raise ValueError(f"Feed forward dimension {d_ff} must be divisible by model dimension {d_model}")
        self.linear1 = torch.nn.Linear(d_model, d_ff)
        self.linear2 = torch.nn.Linear(d_ff, d_model)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.nn.functional.relu(self.linear1(x))
        x = self.linear2(self.dropout(x))
        return x

class EncoderLayer(torch.nn.Module):
    def __init__(self, mha: MultiheadAttention, ffn: FeedForward, dropout: float = 0.1) -> None:
        super().__init__()
        self.attention = deepcopy(mha)
        self.ffn = deepcopy(ffn)
        self.layernorm1 = torch.nn.LayerNorm(mha.d_model)
        self.layernorm2 = torch.nn.LayerNorm(mha.d_model)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        x_norm = self.layernorm1(x)
        x = x + self.attention(x_norm, x_norm, x_norm, mask)[0]
        x_norm = self.layernorm2(x)
        x = self.dropout(x + self.ffn(x_norm))
        return x

class DecoderLayer(torch.nn.Module):
    def __init__(self, mha: MultiheadAttention, enc_dec_mha: MultiheadAttention, ffn: FeedForward, dropout: float = 0.1) -> None:
        super().__init__()
        self.attention1 = deepcopy(mha)
        self.attention2 = deepcopy(enc_dec_mha)
        self.ffn = deepcopy(ffn)
        self.layernorm1 = torch.nn.LayerNorm(mha.d_model)
        self.layernorm2 = torch.nn.LayerNorm(mha.d_model)
        self.layernorm3 = torch.nn.LayerNorm(mha.d_model)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, encoder_memory: torch.Tensor, src_mask: Optional[torch.Tensor], tgt_mask: Optional[torch.Tensor]) -> torch.Tensor:
        x_norm = self.layernorm1(x)
        x = x + self.attention1(x_norm, x_norm, x_norm, tgt_mask)[0]

        # Encoder-decoder attention с маской энкодера
        x_norm = self.layernorm2(x)
        x = x + self.attention2(x_norm, encoder_memory, encoder_memory, src_mask)[0]

        # Feed-forward с маской декодера
        x_norm = self.layernorm3(x)
        x = self.dropout(x + self.ffn(x_norm))
        return x

class Encoder(torch.nn.Module):
    def __init__(self, enc_layer: EncoderLayer, num_layers: int) -> None:
        super().__init__()
        self.layers = torch.nn.ModuleList([deepcopy(enc_layer) for _ in range(num_layers)])

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, mask)
        return x

class Decoder(torch.nn.Module):
    def __init__(self, dec_layer: DecoderLayer, num_layers: int) -> None:
        super().__init__()
        self.layers = torch.nn.ModuleList([deepcopy(dec_layer) for _ in range(num_layers)])

    def forward(self, x: torch.Tensor, encoder_memory: torch.Tensor, src_mask: Optional[torch.Tensor], tgt_mask: Optional[torch.Tensor]) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, encoder_memory, src_mask, tgt_mask)
        return x

generator_model

---



In [57]:
import torch
import torch.nn as nn
import os
import sys



def get_subsequent_mask(seq_len):
    """Создает треугольную маску, чтобы модель не заглядывала в будущее"""
    mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
    return mask

class DecoderOnlyLayer(nn.Module):
    """Слой декодера БЕЗ cross-attention (только для генерации текста)"""
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.self_attention = MultiheadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)

        self.layernorm1 = nn.LayerNorm(d_model)
        self.layernorm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # 1. Masked Self-Attention
        x_norm = self.layernorm1(x)
        attn_out, _ = self.self_attention(x_norm, x_norm, x_norm, mask)
        x = x + attn_out

        # 2. Feed-Forward
        x_norm = self.layernorm2(x)
        ffn_out = self.ffn(x_norm)
        x = self.dropout(x + ffn_out)
        return x

class GeneratorTransformer(nn.Module):
    """Decoder-only Transformer (подобный GPT) для генерации текста"""
    def __init__(
            self,
            vocab_size: int,
            tokenizer,
            d_model: int = 256,
            num_heads: int = 8,
            d_ff: int = 512,
            num_layers: int = 4,
            pad_index: int = 0,
            dropout: float = 0.1,
            max_len: int = 192,
            device: str = 'cpu'
        ):
        super().__init__()
        self.vocab_size = vocab_size
        self.device = device
        self.max_len = max_len
        self.tokenizer = tokenizer

        # Токены (для Mistral tokenizer: eos = 2, pad = 0 (unk))
        self.pad_index = pad_index
        self.eos_token_id = 2

        self.embedding = Embedding(d_model, vocab_size, pad_index)

        self.layers = nn.ModuleList([
            DecoderOnlyLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])

        self.normalize = nn.LayerNorm(d_model)
        self.vocab_projection = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        # Создаем маску для авторегрессии (чтобы токен i не видел токен i+1)
        mask = get_subsequent_mask(seq_len).unsqueeze(0).to(self.device)

        x = self.embedding(x)

        for layer in self.layers:
            x = layer(x, mask)

        x = self.normalize(x)
        logits = self.vocab_projection(x)
        return logits

    def generate(self, prompt: str, context_len: int = 128, temperature: float = 0.8, max_out_tokens: int = 100):
        """Авторегрессивная генерация текста со сдвигом контекста"""
        self.eval()
        with torch.no_grad():
            #  Токенизация промпта
            input_ids = self.tokenizer.encode(prompt).ids
            input_ids = torch.tensor([input_ids], dtype=torch.long).to(self.device)

            generated = input_ids.clone()

            for _ in range(max_out_tokens):
                #  Ограничиваем длину контекста, чтобы не превысить max_len
                current_context = generated[:, -context_len:]

                #  Предсказание
                outputs = self(current_context)

                #  Берем логиты только для последнего токена в последовательности
                next_token_logits = outputs[0, -1, :] / temperature

                #  Сэмплирование
                probs = torch.softmax(next_token_logits, dim=-1)
                next_token = torch.multinomial(probs, 1).unsqueeze(0)

                #  Добавление к сгенерированному тексту
                generated = torch.cat([generated, next_token], dim=1)

                #  Проверка на конец генерации
                if next_token.item() == self.eos_token_id:
                    break

        return self.tokenizer.decode(generated[0].tolist())

    def generate_beam_search(self, prompt: str, beam_size: int = 3, max_out_tokens: int = 50, temperature: float = 0.8):
        """Реализация Beam Search с штрафом за повторение."""
        self.eval()
        with torch.no_grad():
            input_ids = self.tokenizer.encode(prompt).ids
            input_ids = torch.tensor([input_ids], dtype=torch.long).to(self.device)

            # Расширяем входные данные для всех лучей
            sequences = input_ids.repeat(beam_size, 1)
            scores = torch.zeros(beam_size).to(self.device)

            for _ in range(max_out_tokens):
                # Forward pass
                outputs = self(sequences) # [beam_size, seq_len, vocab_size]

                # Берем логиты последнего токена для всех лучей сразу
                next_token_logits = outputs[:, -1, :] / temperature
                penalty = 1.2
                for i in range(beam_size):
                    # Берем токены, которые уже есть в текущем луче i
                    used_tokens = set(sequences[i].tolist())
                    for token in used_tokens:
                        next_token_logits[i, token] /= penalty

                # Считаем log_softmax
                log_probs = torch.log_softmax(next_token_logits, dim=-1)

                # Кумулятивные скоры
                candidate_scores = scores.unsqueeze(1) + log_probs

                # Ищем лучшие варианты
                flat_scores = candidate_scores.view(-1)
                top_scores, top_indices = torch.topk(flat_scores, beam_size)

                # Восстанавливаем индексы
                beam_indices = top_indices // self.vocab_size
                token_indices = top_indices % self.vocab_size

                # Обновляем sequences и scores
                sequences = torch.cat([sequences[beam_indices], token_indices.unsqueeze(1)], dim=1)
                scores = top_scores

            # Возвращаем лучшую последовательность
            best_beam = scores.argmax().item()
            return self.tokenizer.decode(sequences[best_beam].tolist())

    @classmethod
    def load_from_checkpoint(cls, path: str, tokenizer, device: str):
        """Вспомогательный метод для загрузки модели"""
        checkpoint = torch.load(path, map_location=device, weights_only=True)
        config = checkpoint['config']

        model = cls(
            vocab_size=tokenizer.get_vocab_size(),
            tokenizer=tokenizer,
            d_model=config['d_model'],
            num_heads=config['num_heads'],
            d_ff=config['d_ff'],
            num_layers=config['num_layers'],
            max_len=config['max_length'],
            device=device
        )
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
        return model

text_dataset

---



In [58]:
import torch
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
import os

class SlidingWindowTextDataset(Dataset):
    """Датасет, который нарезает длинный текст на окна для обучения авторегрессии"""
    def __init__(self, text_path: str, tokenizer: Tokenizer, max_length: int = 128, stride: int = None):
        self.tokenizer = tokenizer
        self.max_length = max_length
        # Шаг сдвига окна
        self.stride = stride if stride is not None else max_length // 2

        print(f"[*] Чтение и токенизация файла: {text_path}")
        with open(text_path, 'r', encoding='utf-8') as f:
            text = f.read()

        # Токенизируем весь текст целиком
        self.full_tokens = self.tokenizer.encode(text).ids
        print(f"[*] Всего токенов в тексте: {len(self.full_tokens):,}")

        # Создаем индексы для окон
        self.windows = []
        # Мы должны оставить +1 токен для таргета
        for i in range(0, len(self.full_tokens) - max_length, self.stride):
            self.windows.append(i)

        print(f"[*] Создано {len(self.windows):,} блоков контекста")

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        start_idx = self.windows[idx]

        # Берем блок длиной max_length + 1
        chunk = self.full_tokens[start_idx : start_idx + self.max_length + 1]

        input_ids = torch.tensor(chunk[:-1], dtype=torch.long)
        target_ids = torch.tensor(chunk[1:], dtype=torch.long)

        return input_ids, target_ids

def get_text_dataloader(text_path: str, tokenizer_path: str, batch_size: int = 4, max_length: int = 128):
    tokenizer = Tokenizer.from_file(tokenizer_path)

    dataset = SlidingWindowTextDataset(text_path, tokenizer, max_length=max_length)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    return loader, tokenizer

# train_and_chat
 1 - train

---



In [59]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler
from tqdm import tqdm
import os


def train_generator():
    print("=== Обучение текстового генератора ===")

    config = {
        'batch_size': 4,
        'max_length': 128,
        'learning_rate': 3e-4,
        'num_epochs': 20,
        'd_model': 256,
        'num_heads': 8,
        'd_ff': 512,
        'num_layers': 4,
    }

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[*] Используем устройство: {device}")

    # ПУТИ
    text_file = "transformer_basics/training_text.txt"
    tokenizer_file = "transformer_basics/mistral_tokenizer.json"

    if not os.path.exists(text_file):
        print(f"[!] ОШИБКА: Файл {text_file} не найден.")
        print("Пожалуйста, создайте текстовый файл (например, скачайте книгу в формате .txt) и положите по этому пути.")
        return

    # 1. Загрузка данных
    train_loader, tokenizer = get_text_dataloader(text_file, tokenizer_file, config['batch_size'], config['max_length'])
    vocab_size = tokenizer.get_vocab_size()

    # 2. Инициализация модели
    model = GeneratorTransformer(
        vocab_size=vocab_size,
        tokenizer=tokenizer,
        d_model=config['d_model'],
        num_heads=config['num_heads'],
        d_ff=config['d_ff'],
        num_layers=config['num_layers'],
        max_len=config['max_length'],
        device=device
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=config['learning_rate'])
    criterion = nn.CrossEntropyLoss()

    # Инструменты для Mixed Precision
    scaler = GradScaler()

    # Цикл обучения
    for epoch in range(config['num_epochs']):
        model.train()
        total_loss = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['num_epochs']}")
        for input_ids, target_ids in pbar:
            input_ids, target_ids = input_ids.to(device), target_ids.to(device)

            optimizer.zero_grad()

            # --- Mixed Precision Forward ---
            with autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu', dtype=torch.float16):
                logits = model(input_ids) # [batch, seq_len, vocab_size]

                # Переформатируем тензоры для CrossEntropyLoss
                # logits: [batch * seq_len, vocab_size]
                # targets: [batch * seq_len]
                loss = criterion(logits.view(-1, vocab_size), target_ids.view(-1))

            # --- Mixed Precision Backward ---
            scaler.scale(loss).backward()

            # Защита от взрыва градиентов
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        avg_loss = total_loss / len(train_loader)
        print(f"[*] Эпоха {epoch+1} завершена. Средний Loss: {avg_loss:.4f}")

        # Тестовая генерация после каждой эпохи, чтобы видеть прогресс
        print(f"--- Тестовая генерация ---")
        prompt = "The "
        sample = model.generate(prompt, temperature=0.8, max_out_tokens=50)
        print(f"Prompt: '{prompt}' -> {sample}\n")

    # Сохранение чекпоинта
    os.makedirs('transformer_basics/checkpoints', exist_ok=True)
    save_path = 'transformer_basics/checkpoints/generator_model.pt'
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': config
    }, save_path)
    print(f"[*] Модель сохранена в {save_path}")

def chat():
    print("=== Чат с генератором текста ===")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer_file = "transformer_basics/mistral_tokenizer.json"
    checkpoint_path = "transformer_basics/checkpoints/generator_model.pt"

    if not os.path.exists(checkpoint_path):
        print("[!] ОШИБКА: Чекпоинт модели не найден. Сначала запустите обучение!")
        return

    from tokenizers import Tokenizer
    tokenizer = Tokenizer.from_file(tokenizer_file)

    model = GeneratorTransformer.load_from_checkpoint(checkpoint_path, tokenizer, str(device))
    print("[*] Модель успешно загружена! Введите 'quit' для выхода.\n")

    while True:
        user_input = input("Вы: ")
        if user_input.lower() in ['quit', 'exit', 'выход']:
            break

        if not user_input.strip():
            continue

        response = model.generate(
            prompt=user_input,
            context_len=128,
            temperature=0.7,
            max_out_tokens=100
        )
        print(f"Бот: {response}\n")

if __name__ == "__main__":
    # 1. Сначала нужно обучить модель
    train_generator()

    # 2. После обучения закомментировать train_generator() и раскомментировать chat()
    # chat()

=== Обучение текстового генератора ===
[*] Используем устройство: cuda
[*] Чтение и токенизация файла: transformer_basics/training_text.txt
[*] Всего токенов в тексте: 425,186
[*] Создано 6,642 блоков контекста


Epoch 1/20: 100%|██████████| 1660/1660 [00:47<00:00, 34.69it/s, loss=4.6689]


[*] Эпоха 1 завершена. Средний Loss: 5.3981
--- Тестовая генерация ---
Prompt: 'The ' -> The 1. A
      and the room was the expression. I found the
      were the door of this to be give. I heard it? What
      had been held the passenger—and from the subject. But I was given the




Epoch 2/20: 100%|██████████| 1660/1660 [00:48<00:00, 34.10it/s, loss=4.5937]


[*] Эпоха 2 завершена. Средний Loss: 4.5233
--- Тестовая генерация ---
Prompt: 'The ' -> The 1. You can be, or a copy, the
      gossip. Now, you do not be a little
      will be bound to the moor. So something to be at the explanation
      despatch-haired—a



...

[*] Эпоха 18 завершена. Средний Loss: 2.6321
--- Тестовая генерация ---
Prompt: 'The ' -> The 2062702
        Literary Archive Foundation information to the Project Gutenberg Literary Archive Foundation
    
    • You provide a refund of the copyright holder paid a user in the
        eBook. You



Epoch 19/20: 100%|██████████| 1660/1660 [00:48<00:00, 34.31it/s, loss=2.7522]


[*] Эпоха 19 завершена. Средний Loss: 2.5794
--- Тестовая генерация ---
Prompt: 'The ' -> The 2182162618883). Literary. Email contact links at the
additionalow of the full extent permitted by occasion, while the second death of the
trainer and Countess of John Vincent Sp



Epoch 20/20: 100%|██████████| 1660/1660 [00:48<00:00, 34.34it/s, loss=2.0915]


[*] Эпоха 20 завершена. Средний Loss: 2.5298
--- Тестовая генерация ---
Prompt: 'The ' -> The 2_s_.
      diagrams—his uncle’s dead gold-two-James, 1883th30.8823, 187_s_. 18830, Sus

[*] Модель сохранена в transformer_basics/checkpoints/generator_model.pt


2 - chat

---



In [60]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler
from tqdm import tqdm
import os


def train_generator():
    print("=== Обучение текстового генератора ===")

    config = {
        'batch_size': 4, # Увеличили с 1 до 4, так как FP16 экономит память
        'max_length': 128,
        'learning_rate': 3e-4,
        'num_epochs': 3,
        'd_model': 256,
        'num_heads': 8,
        'd_ff': 512,
        'num_layers': 4,
    }

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[*] Используем устройство: {device}")

    text_file = "transformer_basics/training_text.txt"
    tokenizer_file = "transformer_basics/mistral_tokenizer.json"

    if not os.path.exists(text_file):
        print(f"[!] ОШИБКА: Файл {text_file} не найден.")
        print("Пожалуйста, создайте текстовый файл (например, скачайте книгу в формате .txt) и положите по этому пути.")
        return

    # 1. Загрузка данных
    train_loader, tokenizer = get_text_dataloader(text_file, tokenizer_file, config['batch_size'], config['max_length'])
    vocab_size = tokenizer.get_vocab_size()

    # 2. Инициализация модели
    model = GeneratorTransformer(
        vocab_size=vocab_size,
        tokenizer=tokenizer,
        d_model=config['d_model'],
        num_heads=config['num_heads'],
        d_ff=config['d_ff'],
        num_layers=config['num_layers'],
        max_len=config['max_length'],
        device=device
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=config['learning_rate'])
    criterion = nn.CrossEntropyLoss()

    # Инструменты для Mixed Precision
    scaler = GradScaler()

    # Цикл обучения
    for epoch in range(config['num_epochs']):
        model.train()
        total_loss = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['num_epochs']}")
        for input_ids, target_ids in pbar:
            input_ids, target_ids = input_ids.to(device), target_ids.to(device)

            optimizer.zero_grad()

            # --- Mixed Precision Forward ---
            with autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu', dtype=torch.float16):
                logits = model(input_ids) # [batch, seq_len, vocab_size]

                # Переформатируем тензоры для CrossEntropyLoss
                # logits: [batch * seq_len, vocab_size]
                # targets: [batch * seq_len]
                loss = criterion(logits.view(-1, vocab_size), target_ids.view(-1))

            # --- Mixed Precision Backward ---
            scaler.scale(loss).backward()

            # Защита от взрыва градиентов
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        avg_loss = total_loss / len(train_loader)
        print(f"[*] Эпоха {epoch+1} завершена. Средний Loss: {avg_loss:.4f}")

        # Тестовая генерация после каждой эпохи, чтобы видеть прогресс
        print(f"--- Тестовая генерация ---")
        prompt = "The "
        sample = model.generate(prompt, temperature=0.8, max_out_tokens=50)
        print(f"Prompt: '{prompt}' -> {sample}\n")

    # Сохранение чекпоинта
    os.makedirs('transformer_basics/checkpoints', exist_ok=True)
    save_path = 'transformer_basics/checkpoints/generator_model.pt'
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': config
    }, save_path)
    print(f"[*] Модель сохранена в {save_path}")

def chat():
    print("=== Чат с генератором текста ===")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer_file = "transformer_basics/mistral_tokenizer.json"
    checkpoint_path = "transformer_basics/checkpoints/generator_model.pt"

    if not os.path.exists(checkpoint_path):
        print("[!] ОШИБКА: Чекпоинт модели не найден. Сначала запустите обучение!")
        return

    from tokenizers import Tokenizer
    tokenizer = Tokenizer.from_file(tokenizer_file)

    model = GeneratorTransformer.load_from_checkpoint(checkpoint_path, tokenizer, str(device))
    print("[*] Модель успешно загружена! Введите 'quit' для выхода.\n")

    while True:
        user_input = input("Вы: ")
        if user_input.lower() in ['quit', 'exit', 'выход']:
            break

        if not user_input.strip():
            continue

        response = model.generate(
            prompt=user_input,
            context_len=128,
            temperature=0.7,
            max_out_tokens=100
        )
        print(f"Бот: {response}\n")

if __name__ == "__main__":
    # 1. Сначала нужно обучить модель
    #train_generator()

    # 2. После обучения закомментировать train_generator() и раскомментировать chat()
     chat()

=== Чат с генератором текста ===
[*] Модель успешно загружена! Введите 'quit' для выхода.

Вы: Holmes?
Бот: Holmes? You are open, and
      my younger son, that he is a one, he is not dead, and he
      has been in your power.”

      “Have you any other things?”

      “No, sir, he has not always taken the document
      of the letter.”

      “But he has got his right here.”

      “I fancy that the papers?”

      “No, sir, I must give you all the facts

Вы: I need your help
Бот: I need your help, but I am sorry that I should not sleep
matter.”

“She was none of a bit, but I should not quite understand?”

“She was so much surprised for me. I knew that I confess that it was
important to see you. And yet the evidently in my power.

“It is a nice household that I have determined to have had a case of the
breakfastsuggestance,” said Holmes, laughing.

“And

Вы: quit


3 - beam_search

---



In [61]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler
from tqdm import tqdm
import os


def train_generator():
    print("=== Обучение текстового генератора ===")

    config = {
        'batch_size': 4, # Увеличили с 1 до 4, так как FP16 экономит память
        'max_length': 128,
        'learning_rate': 3e-4,
        'num_epochs': 3,
        'd_model': 256,
        'num_heads': 8,
        'd_ff': 512,
        'num_layers': 4,
    }

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[*] Используем устройство: {device}")

    # ПУТИ
    text_file = "transformer_basics/training_text.txt"
    tokenizer_file = "transformer_basics/mistral_tokenizer.json"

    if not os.path.exists(text_file):
        print(f"[!] ОШИБКА: Файл {text_file} не найден.")
        print("Пожалуйста, создайте текстовый файл (например, скачайте книгу в формате .txt) и положите по этому пути.")
        return

    # 1. Загрузка данных
    train_loader, tokenizer = get_text_dataloader(text_file, tokenizer_file, config['batch_size'], config['max_length'])
    vocab_size = tokenizer.get_vocab_size()

    # 2. Инициализация модели
    model = GeneratorTransformer(
        vocab_size=vocab_size,
        tokenizer=tokenizer,
        d_model=config['d_model'],
        num_heads=config['num_heads'],
        d_ff=config['d_ff'],
        num_layers=config['num_layers'],
        max_len=config['max_length'],
        device=device
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=config['learning_rate'])
    criterion = nn.CrossEntropyLoss()

    # Инструменты для Mixed Precision
    scaler = GradScaler()

    # Цикл обучения
    for epoch in range(config['num_epochs']):
        model.train()
        total_loss = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['num_epochs']}")
        for input_ids, target_ids in pbar:
            input_ids, target_ids = input_ids.to(device), target_ids.to(device)

            optimizer.zero_grad()

            # --- Mixed Precision Forward ---
            with autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu', dtype=torch.float16):
                logits = model(input_ids) # [batch, seq_len, vocab_size]

                # Переформатируем тензоры для CrossEntropyLoss
                # logits: [batch * seq_len, vocab_size]
                # targets: [batch * seq_len]
                loss = criterion(logits.view(-1, vocab_size), target_ids.view(-1))

            # --- Mixed Precision Backward ---
            scaler.scale(loss).backward()

            # Защита от взрыва градиентов
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        avg_loss = total_loss / len(train_loader)
        print(f"[*] Эпоха {epoch+1} завершена. Средний Loss: {avg_loss:.4f}")

        # Тестовая генерация после каждой эпохи, чтобы видеть прогресс
        print(f"--- Тестовая генерация ---")
        prompt = "The "
        sample = model.generate(prompt, temperature=0.8, max_out_tokens=50)
        print(f"Prompt: '{prompt}' -> {sample}\n")

    # Сохранение чекпоинта
    os.makedirs('transformer_basics/checkpoints', exist_ok=True)
    save_path = 'transformer_basics/checkpoints/generator_model.pt'
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': config
    }, save_path)
    print(f"[*] Модель сохранена в {save_path}")

def chat():
    print("=== Чат с генератором текста ===")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer_file = "transformer_basics/mistral_tokenizer.json"
    checkpoint_path = "transformer_basics/checkpoints/generator_model.pt"

    if not os.path.exists(checkpoint_path):
        print("[!] ОШИБКА: Чекпоинт модели не найден. Сначала запустите обучение!")
        return

    from tokenizers import Tokenizer
    tokenizer = Tokenizer.from_file(tokenizer_file)

    model = GeneratorTransformer.load_from_checkpoint(checkpoint_path, tokenizer, str(device))
    print("[*] Модель успешно загружена! Введите 'quit' для выхода.\n")

    while True:
        user_input = input("Вы: ")
        if user_input.lower() in ['quit', 'exit', 'выход']:
            break

        if not user_input.strip():
            continue

        response = model.generate_beam_search(
            prompt=user_input,
            beam_size=3,
            max_out_tokens=50
        )
        print(f"Бот: {response}\n")

if __name__ == "__main__":
    # 1. Сначала нужно обучить модель
    #train_generator()

    # 2. После обучения закомментировать train_generator() и раскомментировать chat()
     chat()

=== Чат с генератором текста ===
[*] Модель успешно загружена! Введите 'quit' для выхода.

Вы: help me
Бот: help me to be a very serious felony. I would have made
      the whole affair in my own hands, and I am sure that it is not
      difficult for my purpose.”

      “I will tell you everything,” said Holmes.

Вы: a crime has been commited
Бот: a crime has been commited his own process. He is
      well-known, and I have no doubt that he has not done so much
      neither of the examination. Besides, Watson, it is evident to escape.”

      “But how?”



Вы: quit
